# 04 · Evaluate

Load every artefact, score the **same** evaluation set, and write one comparison table to `../docs/EVALUATION.md`. This is the table ADR-0004 must cite — including any fault the rule detects as well as the forest.

In [ ]:
from pathlib import Path

import pandas as pd

import features

META = ["window_start_ns", "window_end_ns", "label", "fault_kind", "severity", "command_id"]
PARQUET = Path("..") / "data" / f"train-{features.__version__}.parquet"

df = pd.read_parquet(PARQUET)
feature_cols = [c for c in df.columns if c not in META]
X = df[feature_cols].to_numpy(dtype=float)
kinds = df["fault_kind"].tolist()
starts = df["window_start_ns"].to_numpy()
print(f"{len(df)} windows, {len(feature_cols)} features, features v{features.__version__}")

In [ ]:
import numpy as np

# Deterministic time-based split (no RNG), identical in every notebook: the
# earliest 70% of normal windows train the detectors; the rest, plus every
# fault window, form the evaluation set. Training on earlier-normal and
# testing on later-normal is the honest ordering for a live detector.
order = list(np.argsort(starts))
normal_positions = [i for i in order if kinds[i] == "none"]
cut = int(0.7 * len(normal_positions))
train_rows = normal_positions[:cut]
holdout_rows = normal_positions[cut:]
eval_rows = holdout_rows + [i for i in order if kinds[i] != "none"]

X_train = X[train_rows]
X_holdout = X[holdout_rows]
X_eval = X[eval_rows]
kinds_eval = [kinds[i] for i in eval_rows]
print(
    f"train(normal)={len(train_rows)}  eval={len(eval_rows)} "
    f"(holdout normal={len(holdout_rows)}, faults={len(eval_rows) - len(holdout_rows)})"
)

In [ ]:
from detector import evaluate, load_detector, markdown_table

MODELS = Path("..") / "models"
COLUMN = {"rule-v1": "rule", "isoforest-v1": "isoforest"}
results = {}
for stem, column in COLUMN.items():
    detector, manifest = load_detector(stem, MODELS)
    results[column] = evaluate(detector.score(X_eval), kinds_eval, manifest.threshold)
table = markdown_table(results)
print(table)

In [ ]:
n_faults = len(eval_rows) - len(holdout_rows)
intro = (
    "Rule baseline vs Isolation Forest on the twin-anomaly corpus. "
    "Higher recall is better; lower normal-FPR is better. Both "
    f"detectors fit on the same {len(train_rows)} normal-train "
    f"windows and score the same evaluation set ({len(holdout_rows)} "
    f"held-out normal + {n_faults} fault windows), "
    f"features v{features.__version__}."
)
notes = (
    "- **Recall** is per fault kind: the fraction of that fault's "
    "windows flagged anomalous.\n"
    "- **Precision**: of all flagged windows, the fraction truly faulty.\n"
    "- **Normal FPR**: of held-out normal windows, the fraction flagged.\n"
)
tail = (
    "Thresholds come from each artefact's manifest (99th-percentile "
    "normal budget), never service config. Regenerate with "
    "notebooks 02 then 04."
)
doc = f"# Evaluation\n\n{intro}\n\n{table}\n\n{notes}\n{tail}\n"
out = Path("..") / "docs" / "EVALUATION.md"
out.write_text(doc)
print(f"wrote {out}")